# 🕸️ Python Web Scraping
## Module Data Science

> **Prérequis :** Module Python Pur (Chapitres 1 à 5) + NumPy

---

Ce notebook est **interactif ET connecté à Internet** : les cellules ci-dessous envoient de vraies requêtes vers des sites publics et récupèrent des données réelles. Exécutez chaque cellule avec `Shift + Enter`.

⚠️ **Note :** contrairement aux notebooks précédents, les résultats de ce notebook peuvent **varier à chaque exécution** (température du jour, cours de bourse en temps réel...) — c'est normal, c'est même la preuve que le scraping fonctionne !

### Table des matières
1. Explorer le Web Scraping
2. Challenge & Legality of Web Scraping
3. Understanding HTML
4. Getting Started (installation, requests, BeautifulSoup)
5. Scrape HTML Content from a Page
6. Utilizing Classes and IDs
7. Sélecteurs CSS
8. Gérer les erreurs et être un bon scraper
9. Practical Example — Météo (NWS)
10. Practical Example bonus — BRVM
11. Exercices pratiques


In [1]:
# Installation des bibliothèques nécessaires
!pip install requests beautifulsoup4 pandas -q
print("✅ Bibliothèques installées")


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


✅ Bibliothèques installées


In [2]:
# Imports
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

print("✅ Imports effectués")

✅ Imports effectués


---
## 1. Explorer le Web Scraping

Le **Web Scraping** extrait automatiquement des informations depuis des pages web.

```
1. REQUÊTE → 2. ANALYSE → 3. EXTRACTION → 4. STOCKAGE
   requests    BeautifulSoup   .find()      DataFrame
```

---
## 2. Challenge & Legality of Web Scraping

⚠️ Avant de scraper, toujours vérifier :
- Le fichier `robots.txt` du site
- Les Conditions Générales d'Utilisation (CGU)
- La fréquence de vos requêtes (politesse envers le serveur)

In [3]:
import urllib.robotparser

# Vérifier le robots.txt d'un site avant de le scraper
rp = urllib.robotparser.RobotFileParser()
rp.set_url("https://forecast.weather.gov/robots.txt")
rp.read()

url_test = "https://forecast.weather.gov/MapClick.php?lat=37.7772&lon=-122.4168"
peut_scraper = rp.can_fetch("*", url_test)
print(f"Autorisé à scraper cette URL ? {peut_scraper}")

Autorisé à scraper cette URL ? True


---
## 3. Understanding HTML for Web Scraping

Le HTML est un **arbre** : chaque balise peut contenir d'autres balises.
- `id="..."` → **unique** dans la page
- `class="..."` → peut être **partagée** par plusieurs éléments

In [4]:
# Petit exemple de HTML pour s'entraîner
html_exemple = """
<div class="produit">
    <span class="nom">Ordinateur portable</span>
    <span class="prix">850000 FCFA</span>
</div>
<div class="produit">
    <span class="nom">Souris sans fil</span>
    <span class="prix">15000 FCFA</span>
</div>
"""

soup_exemple = BeautifulSoup(html_exemple, "html.parser")
print(soup_exemple.prettify())

<div class="produit">
 <span class="nom">
  Ordinateur portable
 </span>
 <span class="prix">
  850000 FCFA
 </span>
</div>
<div class="produit">
 <span class="nom">
  Souris sans fil
 </span>
 <span class="prix">
  15000 FCFA
 </span>
</div>



---
## 4. Getting Started with Web Scraping

- `requests` → récupère la page (comme un navigateur)
- `BeautifulSoup` → analyse et navigue dans le HTML récupéré

In [5]:
# S'identifier auprès du serveur avec un User-Agent
headers = {
    "User-Agent": "Mozilla/5.0 (Bootcamp Data Science - usage pedagogique)"
}
print("Headers prêts :", headers)

Headers prêts : {'User-Agent': 'Mozilla/5.0 (Bootcamp Data Science - usage pedagogique)'}


---
## 5. Scrape HTML Content from a Page

Envoyons une vraie requête vers le site météo du National Weather Service.

In [6]:
url_meteo = "https://forecast.weather.gov/MapClick.php?lat=37.7772&lon=-122.4168"

reponse = requests.get(url_meteo, headers=headers, timeout=10)
print("Statut :", reponse.status_code)   # 200 = succès

if reponse.status_code == 200:
    print("✅ Page récupérée avec succès")
    print(f"Taille du contenu : {len(reponse.text)} caractères")

Statut : 200
✅ Page récupérée avec succès
Taille du contenu : 48646 caractères


In [7]:
# Transformer le HTML en objet BeautifulSoup
soup = BeautifulSoup(reponse.text, "html.parser")

print("Titre de la page :", soup.title.text)

Titre de la page : National Weather Service


---
## 6. Utilizing Classes and IDs for Efficient Web Scraping

`.find()` → premier élément. `.find_all()` → tous les éléments.

⚠️ Piège : `class` est un mot réservé Python → utiliser `class_` avec BeautifulSoup.

In [8]:
# Cibler le conteneur principal des prévisions via son id (unique)
conteneur_meteo = soup.find(id="seven-day-forecast")

# Trouver chaque bloc de prévision
periodes = conteneur_meteo.find_all("div", class_="tombstone-container")
print(f"Nombre de périodes trouvées : {len(periodes)}")

Nombre de périodes trouvées : 9


In [9]:
# Explorer le premier bloc trouvé
print(periodes[0].prettify()[:500])

<div class="tombstone-container">
 <p class="period-name">
  Today
 </p>
 <p>
  <img alt="Today: Partly sunny, with a high near 67. Southwest wind 11 to 16 mph, with gusts as high as 23 mph. " class="forecast-icon" src="newimages/medium/bkn.png" title="Today: Partly sunny, with a high near 67. Southwest wind 11 to 16 mph, with gusts as high as 23 mph. "/>
 </p>
 <p class="temp temp-high">
  High: 67 °F
 </p>
 <p class="short-desc">
  Partly Sunny
 </p>
</div>



In [10]:
# Extraire le nom de chaque période
noms_periodes = [p.find("p", class_="period-name").text.strip() for p in periodes]
print(noms_periodes)

['Today', 'Tonight', 'Friday', 'Friday Night', 'Saturday', 'Saturday Night', 'Sunday', 'Sunday Night', 'Monday']


---
## 7. Sélecteurs CSS — L'approche moderne

`.select()` utilise directement la syntaxe CSS (`.classe`, `#id`).

In [11]:
# Équivalent avec .select() (sélecteurs CSS)
periodes_css = soup.select("#seven-day-forecast .tombstone-container")
print(f"Nombre trouvé avec .select() : {len(periodes_css)}")

noms_css = [p.select_one(".period-name").text.strip() for p in periodes_css]
print(noms_css)

# Les deux approches donnent le même résultat
print("Résultats identiques :", noms_periodes == noms_css)

Nombre trouvé avec .select() : 9
['Today', 'Tonight', 'Friday', 'Friday Night', 'Saturday', 'Saturday Night', 'Sunday', 'Sunday Night', 'Monday']
Résultats identiques : True


---
## 8. Gérer les erreurs et être un bon scraper

In [12]:
def recuperer_page(url, headers=None):
    """Récupère une page de façon sécurisée, avec gestion d'erreurs."""
    try:
        reponse = requests.get(url, headers=headers, timeout=10)
        reponse.raise_for_status()
        return BeautifulSoup(reponse.text, "html.parser")
    except requests.exceptions.Timeout:
        print("❌ Le serveur a mis trop de temps à répondre")
    except requests.exceptions.HTTPError as erreur:
        print(f"❌ Erreur HTTP : {erreur}")
    except requests.exceptions.RequestException as erreur:
        print(f"❌ Erreur de connexion : {erreur}")
    return None

# Test avec une URL valide
test_soup = recuperer_page(url_meteo, headers=headers)
print("Page récupérée :", test_soup is not None)

# Test avec une URL invalide (pour voir la gestion d'erreur en action)
test_erreur = recuperer_page("https://cette-page-nexiste-pas-12345.com", headers=headers)
print("Résultat attendu (None) :", test_erreur)

Page récupérée : True


❌ Erreur de connexion : HTTPSConnectionPool(host='cette-page-nexiste-pas-12345.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='cette-page-nexiste-pas-12345.com', port=443): Failed to resolve 'cette-page-nexiste-pas-12345.com' ([Errno 8] nodename nor servname provided, or not known)"))
Résultat attendu (None) : None


---
## 9. Practical Example — Prévisions météo (NWS)

Construisons le pipeline complet : requête → parsing → extraction → nettoyage → DataFrame.

In [13]:
# Étape 1-2 : récupérer et parser (déjà fait plus haut, on refait proprement)
url_meteo = "https://forecast.weather.gov/MapClick.php?lat=37.7772&lon=-122.4168"
soup = recuperer_page(url_meteo, headers=headers)

# Étape 3 : cibler le conteneur
conteneur_meteo = soup.find(id="seven-day-forecast")
periodes = conteneur_meteo.find_all("div", class_="tombstone-container")

print(f"Nombre de périodes de prévision trouvées : {len(periodes)}")

Nombre de périodes de prévision trouvées : 9


In [14]:
# Étape 4 : extraire les données de chaque période
donnees_meteo = []

for periode in periodes:
    nom_element = periode.find("p", class_="period-name")
    nom = nom_element.text.strip() if nom_element else "N/A"

    desc_element = periode.find("p", class_="short-desc")
    description = desc_element.text.strip() if desc_element else "N/A"

    temp_element = periode.find("p", class_=lambda c: c and "temp" in c)
    temperature = temp_element.text.strip() if temp_element else "N/A"

    donnees_meteo.append({
        "periode"     : nom,
        "description" : description,
        "temperature" : temperature
    })

for entree in donnees_meteo:
    print(entree)

{'periode': 'Today', 'description': 'Partly Sunny', 'temperature': 'High: 67 °F'}
{'periode': 'Tonight', 'description': 'PatchyDrizzle', 'temperature': 'Low: 58 °F'}
{'periode': 'Friday', 'description': 'PatchyDrizzle thenPartly Sunny', 'temperature': 'High: 66 °F'}
{'periode': 'Friday Night', 'description': 'Mostly Cloudy', 'temperature': 'Low: 58 °F'}
{'periode': 'Saturday', 'description': 'Partly Sunny', 'temperature': 'High: 66 °F'}
{'periode': 'Saturday Night', 'description': 'Mostly Cloudy', 'temperature': 'Low: 57 °F'}
{'periode': 'Sunday', 'description': 'Partly Sunny', 'temperature': 'High: 66 °F'}
{'periode': 'Sunday Night', 'description': 'Mostly Cloudy', 'temperature': 'Low: 57 °F'}
{'periode': 'Monday', 'description': 'Partly Sunny', 'temperature': 'High: 66 °F'}


In [15]:
# Étape 5 : structurer avec Pandas et nettoyer
df_meteo = pd.DataFrame(donnees_meteo)

# Extraire uniquement le nombre de la température
df_meteo["temp_valeur"] = df_meteo["temperature"].str.extract(r"(\d+)").astype(float)
df_meteo["temp_type"]   = df_meteo["temperature"].apply(
    lambda x: "Max" if "High" in x else ("Min" if "Low" in x else "N/A")
)

df_meteo

,periode,description,temperature,temp_valeur,temp_type
0,Today,Partly Sunny,High: 67 °F,67.0,Max
1,Tonight,PatchyDrizzle,Low: 58 °F,58.0,Min
2,Friday,PatchyDrizzle thenPartly Sunny,High: 66 °F,66.0,Max
3,Friday Night,Mostly Cloudy,Low: 58 °F,58.0,Min
4,Saturday,Partly Sunny,High: 66 °F,66.0,Max
5,Saturday Night,Mostly Cloudy,Low: 57 °F,57.0,Min
6,Sunday,Partly Sunny,High: 66 °F,66.0,Max
7,Sunday Night,Mostly Cloudy,Low: 57 °F,57.0,Min
8,Monday,Partly Sunny,High: 66 °F,66.0,Max


In [16]:
# Étape 6 : une petite analyse
temp_max = df_meteo[df_meteo["temp_type"] == "Max"]["temp_valeur"].max()
print(f"🌡️ Température maximale prévue sur la période : {temp_max}°F")

jours_ensoleilles = df_meteo[df_meteo["description"].str.contains("Sunny", case=False)]
print("\n☀️ Périodes ensoleillées :")
print(jours_ensoleilles[["periode", "description"]])

🌡️ Température maximale prévue sur la période : 67.0°F

☀️ Périodes ensoleillées :
    periode                     description
0     Today                    Partly Sunny
2    Friday  PatchyDrizzle thenPartly Sunny
4  Saturday                    Partly Sunny
6    Sunday                    Partly Sunny
8    Monday                    Partly Sunny


In [17]:
# Sauvegarder en CSV
df_meteo.to_csv("previsions_meteo.csv", index=False, encoding="utf-8")
print("✅ Données sauvegardées dans previsions_meteo.csv")

# Vérifier le fichier créé
import os
print("Fichier présent :", os.path.exists("previsions_meteo.csv"))

✅ Données sauvegardées dans previsions_meteo.csv
Fichier présent : True


---
## 10. Practical Example bonus — Cours de la BRVM

Deuxième exemple : les cours de la Bourse Régionale des Valeurs Mobilières (Afrique de l'Ouest).

⚠️ Les résultats varient selon le jour et l'heure de la séance boursière.

In [1]:
# Méthode rapide : pandas.read_html() lit directement les <table> HTML
#
# ⚠️ Piège fréquent : appeler pd.read_html(url_brvm) DIRECTEMENT échoue souvent
# (DataFrame vide, tableau manquant, erreur SSL...) car pandas va chercher la
# page lui-même via urllib, sans les en-têtes d'un vrai navigateur. On récupère
# donc la page nous-mêmes avec requests (+ headers), puis on donne le TEXTE
# HTML (dans io.StringIO) à read_html — jamais l'URL brute.
import io

url_brvm = "https://www.brvm.org/fr/cours-actions/0"

try:
    reponse_brvm = requests.get(url_brvm, headers=headers, timeout=10)
    reponse_brvm.raise_for_status()

    tableaux = pd.read_html(io.StringIO(reponse_brvm.text))
    print(f"Nombre de tableaux trouvés sur la page : {len(tableaux)}")

    # Aperçu de chaque tableau pour identifier celui des cours
    for i, tab in enumerate(tableaux):
        print(f"\n--- Tableau {i} (colonnes : {list(tab.columns)[:5]}...) ---")
        print(tab.head(2))
except Exception as e:
    print(f"❌ Erreur lors de la lecture : {e}")
    print("💡 Le site est peut-être temporairement indisponible, réessayez plus tard.")

❌ Erreur lors de la lecture : name 'requests' is not defined
💡 Le site est peut-être temporairement indisponible, réessayez plus tard.


**👀 À vous d'observer :** repérez dans les résultats ci-dessus l'indice du tableau qui contient les colonnes `Symbole`, `Nom`, `Cours Clôture`, `Variation`. Utilisez cet indice dans la cellule suivante.

In [19]:
# ✏️ Remplacez 0 par l'indice du bon tableau identifié ci-dessus
INDICE_TABLEAU = 0

df_brvm = tableaux[INDICE_TABLEAU]
print(df_brvm.head(10))
print("\nColonnes disponibles :", list(df_brvm.columns))

Empty DataFrame
Columns: [Top 5, Cours, Variation]
Index: []

Colonnes disponibles : ['Top 5', 'Cours', 'Variation']


In [20]:
# Nettoyer une colonne de variation (%) — à adapter selon le nom réel de la colonne
# Exemple générique de nettoyage de données financières scrapées

def nettoyer_nombre(valeur):
    """Convertit '12 155' ou '3,28%' en valeur numérique."""
    if pd.isna(valeur):
        return None
    texte = str(valeur).replace(" ", "").replace("\xa0", "").replace("%", "").replace(",", ".")
    try:
        return float(texte)
    except ValueError:
        return None

# Repérez le nom exact de la colonne de variation dans df_brvm.columns et adaptez ci-dessous
# colonne_variation = "Variation (%)"
# df_brvm["variation_num"] = df_brvm[colonne_variation].apply(nettoyer_nombre)
# print(df_brvm.sort_values("variation_num", ascending=False).head(5))

print("💡 Adaptez le nom de colonne ci-dessus selon ce que vous observez dans df_brvm.columns")

💡 Adaptez le nom de colonne ci-dessus selon ce que vous observez dans df_brvm.columns


---
## 11. 💻 Exercices pratiques

Complétez chaque cellule "✏️ À vous de jouer !", puis comparez avec la solution.

### Exercice 1 — Requête et statut
Envoyez une requête GET vers l'URL météo, affichez le code de statut.

In [21]:
# ✏️ À vous de jouer !



In [22]:
# ✅ Solution Exercice 1
url = "https://forecast.weather.gov/MapClick.php?lat=37.7772&lon=-122.4168"
headers = {"User-Agent": "Bootcamp Data Science - usage pedagogique"}

reponse = requests.get(url, headers=headers, timeout=10)
print("Statut :", reponse.status_code)

if reponse.status_code == 200:
    print("✅ Page récupérée avec succès")

Statut : 200
✅ Page récupérée avec succès


### Exercice 2 — Parsing et titre
Créez un objet BeautifulSoup et affichez le titre de la page.

In [23]:
# ✏️ À vous de jouer !



In [24]:
# ✅ Solution Exercice 2
soup = BeautifulSoup(reponse.text, "html.parser")
print(soup.title.text)

National Weather Service


### Exercice 3 — Extraction ciblée
Trouvez tous les `<p class="period-name">` et affichez leur texte.

In [25]:
# ✏️ À vous de jouer !



In [26]:
# ✅ Solution Exercice 3
periodes = soup.find_all("p", class_="period-name")
for p in periodes:
    print(p.text.strip())

Today
Tonight
Friday
Friday Night
Saturday
Saturday Night
Sunday
Sunday Night
Monday


### Exercice 4 — Sélecteurs CSS
Refaites l'exercice 3 avec `.select()` au lieu de `.find_all()`.

In [27]:
# ✏️ À vous de jouer !



In [28]:
# ✅ Solution Exercice 4
periodes_css = soup.select("p.period-name")
for p in periodes_css:
    print(p.text.strip())

Today
Tonight
Friday
Friday Night
Saturday
Saturday Night
Sunday
Sunday Night
Monday


### Exercice 5 — Pipeline complet
Construisez une fonction `scraper_meteo(url)` qui retourne un DataFrame avec `periode`, `description`, `temperature`.

In [29]:
# ✏️ À vous de jouer !



In [30]:
# ✅ Solution Exercice 5
def scraper_meteo(url):
    headers = {"User-Agent": "Bootcamp Data Science - usage pedagogique"}
    reponse = requests.get(url, headers=headers, timeout=10)

    if reponse.status_code != 200:
        print(f"❌ Erreur : statut {reponse.status_code}")
        return None

    soup = BeautifulSoup(reponse.text, "html.parser")
    conteneur = soup.find(id="seven-day-forecast")
    periodes = conteneur.find_all("div", class_="tombstone-container")

    donnees = []
    for p in periodes:
        nom  = p.find("p", class_="period-name")
        desc = p.find("p", class_="short-desc")
        temp = p.find("p", class_=lambda c: c and "temp" in c)
        donnees.append({
            "periode"     : nom.text.strip() if nom else "N/A",
            "description" : desc.text.strip() if desc else "N/A",
            "temperature" : temp.text.strip() if temp else "N/A"
        })

    return pd.DataFrame(donnees)

df_test = scraper_meteo("https://forecast.weather.gov/MapClick.php?lat=37.7772&lon=-122.4168")
df_test

,periode,description,temperature
0,Today,Partly Sunny,High: 67 °F
1,Tonight,PatchyDrizzle,Low: 58 °F
2,Friday,PatchyDrizzle thenPartly Sunny,High: 66 °F
3,Friday Night,Mostly Cloudy,Low: 58 °F
4,Saturday,Partly Sunny,High: 66 °F
5,Saturday Night,Mostly Cloudy,Low: 57 °F
6,Sunday,Partly Sunny,High: 66 °F
7,Sunday Night,Mostly Cloudy,Low: 57 °F
8,Monday,Partly Sunny,High: 66 °F


### 🏆 Challenge bonus
En utilisant `pandas.read_html()` sur la page BRVM :
1. Identifiez le tableau des cours des actions
2. Nettoyez la colonne de variation en valeur numérique
3. Affichez les 5 titres à la plus forte variation positive
4. Affichez les 5 titres à la plus forte variation négative

In [31]:
# ✏️ À vous de jouer !



In [32]:
# ✅ Piste de solution Challenge
import io

url_brvm = "https://www.brvm.org/fr/cours-actions/0"

# On récupère nous-mêmes le HTML (cf. cellule ci-dessus) plutôt que de passer
# l'URL directement à pd.read_html().
reponse_brvm = requests.get(url_brvm, headers=headers, timeout=10)
tableaux = pd.read_html(io.StringIO(reponse_brvm.text))

# 1. Identifier le tableau (à adapter selon l'inspection réelle)
for i, t in enumerate(tableaux):
    print(f"Tableau {i} — colonnes : {list(t.columns)}")

# Une fois l'indice identifié :
# df_cours = tableaux[i]
#
# # 2. Nettoyer la variation (adapter le nom de colonne réel)
# df_cours["variation_num"] = (
#     df_cours["Variation (%)"].astype(str)
#     .str.replace(",", ".", regex=False)
#     .astype(float)
# )
#
# # 3 & 4.
# top5_hausse = df_cours.sort_values("variation_num", ascending=False).head(5)
# top5_baisse = df_cours.sort_values("variation_num", ascending=True).head(5)
#
# print("Top 5 hausses :\n", top5_hausse)
# print("\nTop 5 baisses :\n", top5_baisse)

Tableau 0 — colonnes : ['Top 5', 'Cours', 'Variation']
Tableau 1 — colonnes : ['Flop 5', 'Cours', 'Variation']
Tableau 2 — colonnes : ['Activités du marché', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3']
Tableau 3 — colonnes : ['Symbole', 'Nom', 'Volume', 'Cours veille (FCFA)', 'Cours Ouverture (FCFA)', 'Cours Clôture (FCFA)', 'Variation (%)']


---
## 🎉 Félicitations !

Vous maîtrisez maintenant les fondamentaux du Web Scraping avec Python :
- **Légalité et éthique** — `robots.txt`, CGU, politesse envers les serveurs
- **requests + BeautifulSoup** — récupérer et parser du HTML
- **`.find()` / `.find_all()` / `.select()`** — cibler des données précises
- **`pandas.read_html()`** — extraction rapide de tableaux
- **Pipeline complet** — requête → extraction → nettoyage → DataFrame → CSV

**Prochain chapitre :** Pandas en profondeur — structurer, nettoyer et analyser vos données.

*📘 Module Data Science — Python Web Scraping | Bootcamp Data Science*